# Dependencies, functions and configurations

In [1]:
from ollama import chat
import time
import os
import re
import pandas as pd
from dotenv import load_dotenv
from collections import defaultdict
import csv
from datetime import datetime

In [2]:
load_dotenv()
the_wizard = os.getenv("WIZARD")
the_spell = os.getenv("LEVIOSA")

In [3]:
def log_performance(start_time, end_time, image_count, input_tokens, output_tokens, log_csv='performance_log_2.csv'):
    duration = end_time - start_time
    log_data = {
        'date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'image_count': image_count,
        'duration_seconds': round(duration, 2),
        'input_tokens': input_tokens,
        'output_tokens': output_tokens
    }

    file_exists = os.path.isfile(log_csv)
    with open(log_csv, 'a', newline='') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=log_data.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(log_data)
    print(f"Duration time per image: {round(duration, 2)/image_count}")

In [4]:
def load_image_paths(csv_path):
    return pd.read_csv(csv_path)['image_path'].tolist()

Con la funcion detallada arriba faltara extraer informacion de la imagen nueva cargada manualmente en la carpeta de Colombia, la cual no cuenta con su path en el archivo csv de los paths a las imagenes

In [5]:
def load_processed_paths(processed_csv):
    if os.path.exists(processed_csv):
        return set(pd.read_csv(processed_csv)['path'].tolist())
    return set()

In [6]:
def save_processed_paths(processed_csv, new_paths):
    df = pd.DataFrame({'path': list(new_paths)})
    if os.path.exists(processed_csv):
        existing = pd.read_csv(processed_csv)
        df = pd.concat([existing, df], ignore_index=True).drop_duplicates()
    df.to_csv(processed_csv, index=False)

In [7]:
def extract_data_from_images(image_paths, wizard, spell, content_response):
    total_input_tokens = 0
    total_output_tokens = 0

    for path in image_paths:
        parts = os.path.basename(path).split('_')
        statistic = '_'.join(parts[:-1])
        country = parts[-1].replace('.png', '')

        response = chat(
            model=wizard,
            messages=[
                {
                    'role': 'user',
                    'content': spell,
                    'images': [path],
                }
            ],
        )

        # Acumular tokens
        total_input_tokens += response.prompt_eval_count
        total_output_tokens += response.eval_count

        # Guardar contenido como antes
        if country not in content_response:
            content_response[country] = {}
        content_response[country][statistic] = response.message.content
    print(content_response)
    return total_input_tokens, total_output_tokens


In [8]:
def clean_ollama_response(text):
    # Step 1: Remove all asterisks
    text = text.replace('*', '')
    
    # Step 2: Remove all newline characters
    text = text.replace('\n', '')
    
    # Step 3: Remove everything before and including the first colon
    colon_index = text.find(':')
    if colon_index != -1:
        text = text[colon_index + 1:]
    
    # Optional: Strip leading/trailing whitespace
    return text.strip()

In [9]:
def build_statistic_dataframes(data):
    """
    Builds a dictionary of DataFrames, one per statistic.
    Each DataFrame has years as index and countries as columns.
    Missing values are filled with NaN.
    """
    stats_data = defaultdict(lambda: defaultdict(dict))  # {stat: {year: {country: value}}}
    all_countries = set(data.keys())  # Track all countries from input

    for country, stats in data.items():
        for stat_name, stat_text in stats.items():
            cleaned = clean_ollama_response(stat_text)
            matches = re.findall(r"(\d{4}):\s*(-?[\d.]+)", cleaned)
            for year_str, value_str in matches:
                try:
                    year = int(year_str)
                    value = float(value_str)
                    stats_data[stat_name][year][country] = value
                except ValueError:
                    continue  # Skip malformed entries

    stat_dfs = {}
    for stat_name, year_country_values in stats_data.items():
        df = pd.DataFrame.from_dict(year_country_values, orient='index').sort_index()
        df = df.reindex(columns=all_countries)  # Ensure all countries are present
        stat_dfs[stat_name] = df

    return stat_dfs

In [10]:
# si trabajamos sobre varias imagenes tendremos un solo diccionario con varios paises y varios estadisticos
# necesitmos una forma de serpararlos y definir un dataframe por cada estditico y pais
def export_statistic_dataframes_to_parquet(stat_dfs, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    for name, new_df in stat_dfs.items():
        output_path = os.path.join(output_dir, f"{name}.parquet")

        if os.path.exists(output_path):
            existing_df = pd.read_parquet(output_path)

            # Merge on index (years) and columns (countries)
            combined_df = existing_df.combine_first(new_df)
        else:
            combined_df = new_df

        combined_df.to_parquet(output_path, index=True)
        print(f"✅ Merged: {output_path}")

In [11]:
def run_extraction_pipeline(
    input_csv,
    processed_csv,
    model=the_wizard,
    spell=the_spell,
    batch_size=4,
    output_dir='parquet_exports_2'
):
    start_time = time.time()

    # Step 1: Load paths
    all_paths = load_image_paths(input_csv)
    processed_paths = load_processed_paths(processed_csv)
    remaining_paths = [p for p in all_paths if p not in processed_paths]

    if not remaining_paths:
        print("✅ All images have already been processed. Nothing to do.")
        return

    # Step 2: Select batch
    batch_paths = remaining_paths[:batch_size]
    print(f"Processing {len(batch_paths)} images...")

    # Step 3: Extract data
    content_response = {}
    input_tokens, output_tokens = extract_data_from_images(batch_paths, model, spell, content_response)

    # Step 4: Save processed paths
    save_processed_paths(processed_csv, batch_paths)

    # Step 5: Build DataFrames and export
    stat_dfs = build_statistic_dataframes(content_response)
    export_statistic_dataframes_to_parquet(stat_dfs, output_dir=output_dir)

    end_time = time.time()

    # Step 7: Log performance extendido
    log_performance(
        start_time,
        end_time,
        image_count=len(batch_paths),
        input_tokens=input_tokens,
        output_tokens=output_tokens
    )

    print(f"The process for {len(batch_paths)} images has finished.")

In [12]:
# para cargar todos los parquet files en un directorio seleccionado
def load_parquet_files(directory):
    dataframes = {}
    for filename in os.listdir(directory):
        if filename.endswith(".parquet"):
            path = os.path.join(directory, filename)
            df = pd.read_parquet(path)
            dataframes[filename] = df
    return dataframes

In [13]:
# para observar si es que encontramos una incongruencia en la extraccion del LLM
def view_country_statistic(parquet_data, statistic_name, country_name):
    df = parquet_data.get(f"{statistic_name}.parquet")
    if df is None:
        print(f"❌ Statistic '{statistic_name}' not found.")
        return

    if country_name not in df.columns:
        print(f"❌ Country '{country_name}' not found in '{statistic_name}'.")
        return

    print(f"\n📊 {statistic_name} for {country_name}")
    print(df[[country_name]].dropna())

In [14]:
paths_to_image = pd.read_csv("../data/image_paths_2.csv")

In [15]:
paths_to_image.shape

(337, 1)

### Primera iteracion de 90 imagenes

- se tendra en cuenta los numeros negativos
- anteriormente no se tenian en cuenta los mismos en el patron capturado por la libreria *re*
- si se toma 60 segundos por cada imagen segun los logs entonces podre ir al gym 90min es decir 1hr y 30min
- quedaran por extraer 337 - 90 = 247 imagenes

In [16]:
run_extraction_pipeline(
    input_csv='../data/image_paths_2.csv',
    processed_csv='../data/processed_paths_2.csv',
    batch_size=90
)

Processing 90 images...
{'Australia': {'Education_spending': 'Aquí tienes los años y valores extraídos del gráfico:\n\n*   **2015:** 5.46\n*   **2016:** 5.43\n*   **2017:** 5.32\n*   **2018:** 5.41\n*   **2019:** 5.51\n*   **2020:** 5.38\n*   **2021:** 5.33\n*   **2022:** 5.21', 'Inflation': 'Aquí están los años y valores extraídos del gráfico:\n\n*   **2017:** 1.90\n*   **2018:** 1.90\n*   **2019:** 1.60\n*   **2020:** 0.80\n*   **2021:** 2.90\n*   **2022:** 6.60\n*   **2023:** 5.60\n*   **2024:** 3.20', 'Share_of_industry': 'Aquí están los años y valores extraídos de la imagen:\n\n*   **2017:** 23.47\n*   **2018:** 24.12\n*   **2019:** 25.27\n*   **2020:** 25.39\n*   **2021:** 25.42\n*   **2022:** 27.48\n*   **2023:** 27.65\n*   **2024:** 25.97', 'Tax_revenue': 'Aquí están los años y valores extraídos del gráfico que proporcionaste:\n\n*   **2015:** 21.87\n*   **2016:** 22.26\n*   **2017:** 22.09\n*   **2018:** 23.18\n*   **2019:** 23.38\n*   **2020:** 22.56\n*   **2021:** 23.03\n*  

In [17]:
# Load all Parquet files from the output directory
parquet_data = load_parquet_files("parquet_exports_2")

# Inspect each DataFrame
for name, df in parquet_data.items():
    print(f"\n📄 File: {name}")
    print(df.head())


📄 File: Education_spending.parquet
      Brazil  France  Australia  Colombia  Denmark  Chile  Czech-Republic  \
2014    5.95    5.49        NaN       NaN      NaN   4.75            3.67   
2015    6.24    5.45       5.46       NaN     7.01   4.90            5.75   
2016    6.31    5.41       5.43       NaN     7.69   5.37            5.55   
2017    6.32    5.45       5.32       NaN     7.12   5.43            3.81   
2018    6.09    5.41       5.41       NaN     7.30   5.47            4.27   

      Finland  China  Austria  Belgium  Cyprus  Canada  Estonia  Bulgaria  \
2014     7.10    NaN      NaN     6.59    6.42     NaN     4.39      4.07   
2015     7.03   4.24     5.46     6.45    6.30    4.74     5.14      3.91   
2016     6.85   4.21     5.48     6.46    6.12    4.82     5.16      3.39   
2017     6.36   4.11     5.37     6.43    5.70    4.96     4.96      4.08   
2018     6.28   4.02     5.23     6.38    5.03    4.89     5.24      4.05   

      Croatia  
2014     5.18  
2015  